# Day 21 - Investment Recommendation & Decision Engine

## Objectives

- Combine company quality, financial momentum, and risk signals
- Build a normalized investment scoring model
- Generate rule-based investment recommendations
- Calculate recommendation confidence
- Rank companies using final investment scores
- Identify top investment candidates
- Generate final investment decision reports

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

REPORT_DIR = "../reports"

print("Day 21 - Investment Recommendation & Decision Engine")

Day 21 - Investment Recommendation & Decision Engine


In [3]:
import os

report_path = "../reports"

print("FILES IN REPORTS FOLDER:")
print("=" * 60)

for file in sorted(os.listdir(report_path)):
    print(file)

FILES IN REPORTS FOLDER:
benchmark_companies.csv
benchmark_companies_day7.csv
capital_allocation_day11.csv
company_ranking_day13.csv
day14_infinite_values_report.csv
day14_missing_values_report.csv
day14_ratio_edge_cases.csv
day14_validation_summary.csv
day15_filter_summary.csv
day15_screener_output.csv
day16_all_preset_results.csv
day16_balanced_screener.csv
day16_high_roe_screener.csv
day16_income_screener.csv
day16_low_debt_screener.csv
day16_preset_summary.csv
day16_quality_screener.csv
day17_composite_scores.csv
day17_score_band_summary.csv
day17_top20_companies.csv
day18_benchmark_peer_performance.csv
day18_peer_comparison.csv
day18_peer_leaders.csv
day18_peer_percentiles.csv
day19_declining_companies.csv
day19_financial_trends.csv
day19_latest_company_trends.csv
day19_top_improving_companies.csv
day19_trend_distribution.csv
day20_company_risk_report.csv
day20_high_risk_companies.csv
day20_low_risk_companies.csv
day20_risk_distribution.csv
day20_risk_driver_distribution.csv
finan

In [4]:
import glob

print("POSSIBLE DAY 17 SCORE FILES:")
print("=" * 60)

possible_files = []

for file in glob.glob("../reports/*.csv"):
    name = os.path.basename(file).lower()

    if (
        "rank" in name
        or "score" in name
        or "composite" in name
        or "day17" in name
    ):
        possible_files.append(file)
        print(file)

POSSIBLE DAY 17 SCORE FILES:
../reports\company_ranking_day13.csv
../reports\day17_composite_scores.csv
../reports\day17_score_band_summary.csv
../reports\day17_top20_companies.csv


In [5]:
score_path = "../reports/day17_composite_scores.csv"

scores = pd.read_csv(score_path)

print("COMPOSITE SCORE DATA")
print("Shape:", scores.shape)

print("\nColumns:")
print(scores.columns.tolist())

scores.head()

COMPOSITE SCORE DATA
Shape: (92, 16)

Columns:
['company_id', 'year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio', 'year_number', 'roe_score', 'debt_score', 'interest_score', 'asset_turnover_score', 'dividend_score', 'composite_score', 'company_rank', 'score_band']


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,year_number,roe_score,debt_score,interest_score,asset_turnover_score,dividend_score,composite_score,company_rank,score_band
0,BEL,Mar 2024,4744.047619,0.511905,420.916667,125.888199,1.003764,2024,100.000000,96.556869,15.685033,100.000000,9.131016,73.189325,1,Strong
1,HAL,Mar 2024,3816.582915,0.618090,226.720930,67.513333,0.408163,2024,80.471889,95.842652,8.447280,53.624880,3.712969,58.206714,2,Average
2,BAJFINANCE,Mar 2024,18.841921,3.824789,2683.166667,0.146303,0.103799,2024,0.509052,74.274025,100.000000,0.106055,0.944236,38.831554,3,Weak
3,INDIGO,Mar 2024,892.568306,0.018579,22.602322,56.386252,0.000000,2024,18.905684,99.875034,0.839698,44.785121,0.000000,37.526171,4,Weak
4,BHEL,Mar 2024,1.153941,0.362386,0.858696,0.399629,10.992908,2024,0.136625,97.562546,0.029304,0.307307,100.000000,34.483581,5,Weak


In [6]:
trend_path = "../reports/day19_latest_company_trends.csv"

trends = pd.read_csv(trend_path)

print("TREND DATA")
print("Shape:", trends.shape)

print("\nColumns:")
print(trends.columns.tolist())

trends.head()

TREND DATA
Shape: (92, 19)

Columns:
['company_id', 'year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio', 'financial_year', 'return_on_equity_pct_yoy_change', 'debt_to_equity_yoy_change', 'interest_coverage_yoy_change', 'asset_turnover_yoy_change', 'dividend_payout_ratio_yoy_change', 'roe_momentum', 'debt_momentum', 'interest_momentum', 'asset_turnover_momentum', 'financial_momentum_score', 'trend_classification']


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio,financial_year,return_on_equity_pct_yoy_change,debt_to_equity_yoy_change,interest_coverage_yoy_change,asset_turnover_yoy_change,dividend_payout_ratio_yoy_change,roe_momentum,debt_momentum,interest_momentum,asset_turnover_momentum,financial_momentum_score,trend_classification
0,ABB,Mar 2024,32.468235,0.022438,121.083333,1.126324,6.078268,2024,2.700355,-0.013007,45.708333,-0.047732,-1.614040,1,0,1,0,2,Improving
1,ADANIENSOL,Mar 2024,9.461277,2.932521,2.063968,0.283696,0.000000,2024,-1.441779,0.015677,0.439373,0.037219,0.000000,0,0,0,0,0,Declining
2,ADANIENT,Mar 2024,8.534650,1.671358,2.497695,0.600432,0.149925,2024,1.206582,0.061725,0.275977,-0.302327,-0.097804,0,0,0,-1,-1,Declining
3,ADANIGREEN,Mar 2024,12.812691,6.595282,1.461846,0.104670,0.000000,2024,-0.508777,-0.828459,-0.245471,-0.011547,0.000000,0,1,0,0,1,Improving
4,ADANIPORTS,Mar 2024,15.354883,0.937322,5.703988,0.228301,0.197433,2024,3.477556,-0.239923,1.071318,0.043054,-0.173555,1,1,1,0,3,Strong Improvement


In [7]:
risk_path = "../reports/day20_company_risk_report.csv"

risks = pd.read_csv(risk_path)

print("RISK DATA")
print("Shape:", risks.shape)

print("\nColumns:")
print(risks.columns.tolist())

risks.head()

RISK DATA
Shape: (92, 17)

Columns:
['company_id', 'financial_year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'financial_momentum_score', 'debt_risk_score', 'profitability_risk_score', 'interest_risk_score', 'efficiency_risk_score', 'momentum_risk_score', 'total_risk_score', 'normalized_risk_score', 'risk_category', 'risk_rank', 'primary_risk_driver']


,company_id,financial_year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,financial_momentum_score,debt_risk_score,profitability_risk_score,interest_risk_score,efficiency_risk_score,momentum_risk_score,total_risk_score,normalized_risk_score,risk_category,risk_rank,primary_risk_driver
0,KOTAKBANK,2024,14.013018,4.003739,1.237006,0.073257,-1,3,1,2,3,2,11,73.33,High Risk,1,Debt Risk
1,SHRIRAMFIN,2024,15.116350,3.994034,0.675071,0.146569,-1,3,0,3,3,2,11,73.33,High Risk,1,Debt Risk
2,UNIONBANK,2024,14.136560,12.823705,0.072439,0.071595,2,3,1,3,3,1,11,73.33,High Risk,1,Debt Risk
3,ADANIGREEN,2024,12.812691,6.595282,1.461846,0.104670,1,3,1,2,3,1,10,66.67,High Risk,2,Debt Risk
4,HDFCBANK,2024,14.339740,6.808787,1.400897,0.070381,1,3,1,2,3,1,10,66.67,High Risk,2,Debt Risk


In [8]:
print("=" * 60)
print("SCORE COLUMNS")
print("=" * 60)

for col in scores.columns:
    print(col)

print("\n" + "=" * 60)
print("TREND COLUMNS")
print("=" * 60)

for col in trends.columns:
    print(col)

print("\n" + "=" * 60)
print("RISK COLUMNS")
print("=" * 60)

for col in risks.columns:
    print(col)

SCORE COLUMNS
company_id
year
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
dividend_payout_ratio
year_number
roe_score
debt_score
interest_score
asset_turnover_score
dividend_score
composite_score
company_rank
score_band

TREND COLUMNS
company_id
year
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
dividend_payout_ratio
financial_year
return_on_equity_pct_yoy_change
debt_to_equity_yoy_change
interest_coverage_yoy_change
asset_turnover_yoy_change
dividend_payout_ratio_yoy_change
roe_momentum
debt_momentum
interest_momentum
asset_turnover_momentum
financial_momentum_score
trend_classification

RISK COLUMNS
company_id
financial_year
return_on_equity_pct
debt_to_equity
interest_coverage
asset_turnover
financial_momentum_score
debt_risk_score
profitability_risk_score
interest_risk_score
efficiency_risk_score
momentum_risk_score
total_risk_score
normalized_risk_score
risk_category
risk_rank
primary_risk_driver


In [9]:
possible_score_columns = [
    "composite_score",
    "final_score",
    "normalized_score",
    "investment_score"
]

score_column = next(
    (
        col
        for col in possible_score_columns
        if col in scores.columns
    ),
    None
)

print("Detected Score Column:", score_column)

Detected Score Column: composite_score


In [10]:
score_data = scores[
    [
        "company_id",
        score_column
    ]
].copy()

trend_data = trends[
    [
        "company_id",
        "financial_momentum_score",
        "trend_classification"
    ]
].copy()

risk_data = risks[
    [
        "company_id",
        "normalized_risk_score",
        "risk_category",
        "primary_risk_driver"
    ]
].copy()

print("Score Data:", score_data.shape)
print("Trend Data:", trend_data.shape)
print("Risk Data:", risk_data.shape)

Score Data: (92, 2)
Trend Data: (92, 3)
Risk Data: (92, 4)


In [11]:
score_data = score_data.drop_duplicates(
    subset=["company_id"]
)

trend_data = trend_data.drop_duplicates(
    subset=["company_id"]
)

risk_data = risk_data.drop_duplicates(
    subset=["company_id"]
)

print("Unique Score Companies:", score_data["company_id"].nunique())
print("Unique Trend Companies:", trend_data["company_id"].nunique())
print("Unique Risk Companies:", risk_data["company_id"].nunique())

Unique Score Companies: 92
Unique Trend Companies: 92
Unique Risk Companies: 92


In [12]:
decision_data = score_data.merge(
    trend_data,
    on="company_id",
    how="inner"
)

decision_data = decision_data.merge(
    risk_data,
    on="company_id",
    how="inner"
)

print("Decision Dataset Shape:", decision_data.shape)

decision_data.head(10)

Decision Dataset Shape: (92, 7)


,company_id,composite_score,financial_momentum_score,trend_classification,normalized_risk_score,risk_category,primary_risk_driver
0,BEL,73.189325,3,Strong Improvement,6.67,Low Risk,Debt Risk
1,HAL,58.206714,0,Declining,13.33,Low Risk,Debt Risk
2,BAJFINANCE,38.831554,1,Improving,46.67,Moderate Risk,Debt Risk
3,INDIGO,37.526171,1,Improving,6.67,Low Risk,Financial Momentum Risk
4,BHEL,34.483581,-1,Declining,66.67,High Risk,Interest Coverage Risk
5,NAUKRI,32.681642,1,Improving,40.00,Moderate Risk,Asset Efficiency Risk
6,DIVISLAB,32.127323,-2,Strong Decline,40.00,Moderate Risk,Financial Momentum Risk
7,ABB,31.765038,2,Improving,13.33,Low Risk,Asset Efficiency Risk
8,BAJAJHLDNG,31.187216,2,Improving,33.33,Moderate Risk,Asset Efficiency Risk
9,TECHM,30.850326,-2,Strong Decline,33.33,Moderate Risk,Financial Momentum Risk


In [13]:
numeric_columns = [
    score_column,
    "financial_momentum_score",
    "normalized_risk_score"
]

for col in numeric_columns:
    decision_data[col] = pd.to_numeric(
        decision_data[col],
        errors="coerce"
    )

print(decision_data[numeric_columns].dtypes)

composite_score             float64
financial_momentum_score      int64
normalized_risk_score       float64
dtype: object


In [14]:
print("Missing Values:")

print(
    decision_data[
        [
            score_column,
            "financial_momentum_score",
            "normalized_risk_score"
        ]
    ].isna().sum()
)

Missing Values:
composite_score             0
financial_momentum_score    0
normalized_risk_score       0
dtype: int64


In [15]:
decision_data["financial_momentum_score"] = (
    decision_data["financial_momentum_score"]
    .fillna(0)
)

decision_data["normalized_risk_score"] = (
    decision_data["normalized_risk_score"]
    .fillna(50)
)

In [16]:
print(
    decision_data[score_column]
    .describe()
)

count    92.000000
mean     24.837597
std       8.956936
min       0.270035
25%      22.836630
50%      25.490683
75%      27.215109
max      73.189325
Name: composite_score, dtype: float64


In [17]:
score_min = decision_data[score_column].min()
score_max = decision_data[score_column].max()

if score_max == score_min:
    decision_data["quality_score_100"] = 50.0
else:
    decision_data["quality_score_100"] = (
        (
            decision_data[score_column] - score_min
        )
        /
        (
            score_max - score_min
        )
    ) * 100

decision_data["quality_score_100"] = (
    decision_data["quality_score_100"]
    .round(2)
)

decision_data[
    [
        "company_id",
        score_column,
        "quality_score_100"
    ]
].head(15)

,company_id,composite_score,quality_score_100
0,BEL,73.189325,100.00
1,HAL,58.206714,79.45
2,BAJFINANCE,38.831554,52.88
3,INDIGO,37.526171,51.09
4,BHEL,34.483581,46.92
5,NAUKRI,32.681642,44.45
6,DIVISLAB,32.127323,43.69
7,ABB,31.765038,43.19
8,BAJAJHLDNG,31.187216,42.40
9,TECHM,30.850326,41.94


In [18]:
momentum_min = decision_data[
    "financial_momentum_score"
].min()

momentum_max = decision_data[
    "financial_momentum_score"
].max()

if momentum_max == momentum_min:
    decision_data["momentum_score_100"] = 50.0
else:
    decision_data["momentum_score_100"] = (
        (
            decision_data["financial_momentum_score"]
            - momentum_min
        )
        /
        (
            momentum_max - momentum_min
        )
    ) * 100

decision_data["momentum_score_100"] = (
    decision_data["momentum_score_100"]
    .round(2)
)

decision_data[
    [
        "company_id",
        "financial_momentum_score",
        "momentum_score_100"
    ]
].head(15)

,company_id,financial_momentum_score,momentum_score_100
0,BEL,3,85.71
1,HAL,0,42.86
2,BAJFINANCE,1,57.14
3,INDIGO,1,57.14
4,BHEL,-1,28.57
5,NAUKRI,1,57.14
6,DIVISLAB,-2,14.29
7,ABB,2,71.43
8,BAJAJHLDNG,2,71.43
9,TECHM,-2,14.29


In [19]:
decision_data["safety_score_100"] = (
    100
    - decision_data["normalized_risk_score"]
)

decision_data["safety_score_100"] = (
    decision_data["safety_score_100"]
    .clip(0, 100)
    .round(2)
)

decision_data[
    [
        "company_id",
        "normalized_risk_score",
        "safety_score_100"
    ]
].head(15)

,company_id,normalized_risk_score,safety_score_100
0,BEL,6.67,93.33
1,HAL,13.33,86.67
2,BAJFINANCE,46.67,53.33
3,INDIGO,6.67,93.33
4,BHEL,66.67,33.33
5,NAUKRI,40.00,60.00
6,DIVISLAB,40.00,60.00
7,ABB,13.33,86.67
8,BAJAJHLDNG,33.33,66.67
9,TECHM,33.33,66.67


In [20]:
QUALITY_WEIGHT = 0.50
MOMENTUM_WEIGHT = 0.20
SAFETY_WEIGHT = 0.30

decision_data["final_investment_score"] = (
    decision_data["quality_score_100"]
    * QUALITY_WEIGHT
    +
    decision_data["momentum_score_100"]
    * MOMENTUM_WEIGHT
    +
    decision_data["safety_score_100"]
    * SAFETY_WEIGHT
)

decision_data["final_investment_score"] = (
    decision_data["final_investment_score"]
    .round(2)
)

decision_data[
    [
        "company_id",
        "quality_score_100",
        "momentum_score_100",
        "safety_score_100",
        "final_investment_score"
    ]
].head(20)

,company_id,quality_score_100,momentum_score_100,safety_score_100,final_investment_score
0,BEL,100.00,85.71,93.33,95.14
1,HAL,79.45,42.86,86.67,74.30
2,BAJFINANCE,52.88,57.14,53.33,53.87
3,INDIGO,51.09,57.14,93.33,64.97
4,BHEL,46.92,28.57,33.33,39.17
5,NAUKRI,44.45,57.14,60.00,51.65
6,DIVISLAB,43.69,14.29,60.00,42.70
7,ABB,43.19,71.43,86.67,61.88
8,BAJAJHLDNG,42.40,71.43,66.67,55.49
9,TECHM,41.94,14.29,66.67,43.83


In [21]:
decision_data["recommendation"] = np.select(
    [
        decision_data["normalized_risk_score"] > 75,

        (
            (decision_data["final_investment_score"] >= 75)
            &
            (decision_data["normalized_risk_score"] <= 50)
        ),

        (
            (decision_data["final_investment_score"] >= 60)
            &
            (decision_data["normalized_risk_score"] <= 60)
        ),

        decision_data["final_investment_score"] >= 40
    ],
    [
        "High Risk",
        "Strong Buy",
        "Buy",
        "Hold"
    ],
    default="Avoid"
)

decision_data[
    [
        "company_id",
        "final_investment_score",
        "normalized_risk_score",
        "recommendation"
    ]
].head(20)

,company_id,final_investment_score,normalized_risk_score,recommendation
0,BEL,95.14,6.67,Strong Buy
1,HAL,74.30,13.33,Buy
2,BAJFINANCE,53.87,46.67,Hold
3,INDIGO,64.97,6.67,Buy
4,BHEL,39.17,66.67,Avoid
5,NAUKRI,51.65,40.00,Hold
6,DIVISLAB,42.70,40.00,Hold
7,ABB,61.88,13.33,Buy
8,BAJAJHLDNG,55.49,33.33,Hold
9,TECHM,43.83,33.33,Hold


In [22]:
risk_override_count = (
    (
        decision_data["normalized_risk_score"] > 75
    )
    &
    (
        decision_data["recommendation"] == "High Risk"
    )
).sum()

print(
    "Companies Protected By Risk Override:",
    risk_override_count
)

Companies Protected By Risk Override: 0


In [23]:
decision_data["recommendation_confidence"] = (
    np.abs(
        decision_data["final_investment_score"]
        - 50
    )
    * 2
)

decision_data["recommendation_confidence"] = (
    decision_data["recommendation_confidence"]
    .clip(0, 100)
    .round(2)
)

decision_data[
    [
        "company_id",
        "final_investment_score",
        "recommendation",
        "recommendation_confidence"
    ]
].head(20)

,company_id,final_investment_score,recommendation,recommendation_confidence
0,BEL,95.14,Strong Buy,90.28
1,HAL,74.30,Buy,48.60
2,BAJFINANCE,53.87,Hold,7.74
3,INDIGO,64.97,Buy,29.94
4,BHEL,39.17,Avoid,21.66
5,NAUKRI,51.65,Hold,3.30
6,DIVISLAB,42.70,Hold,14.60
7,ABB,61.88,Buy,23.76
8,BAJAJHLDNG,55.49,Hold,10.98
9,TECHM,43.83,Hold,12.34


In [24]:
decision_data["confidence_level"] = pd.cut(
    decision_data["recommendation_confidence"],
    bins=[
        -np.inf,
        30,
        60,
        np.inf
    ],
    labels=[
        "Low Confidence",
        "Moderate Confidence",
        "High Confidence"
    ]
)

decision_data[
    [
        "company_id",
        "recommendation",
        "recommendation_confidence",
        "confidence_level"
    ]
].head(20)

,company_id,recommendation,recommendation_confidence,confidence_level
0,BEL,Strong Buy,90.28,High Confidence
1,HAL,Buy,48.60,Moderate Confidence
2,BAJFINANCE,Hold,7.74,Low Confidence
3,INDIGO,Buy,29.94,Low Confidence
4,BHEL,Avoid,21.66,Low Confidence
5,NAUKRI,Hold,3.30,Low Confidence
6,DIVISLAB,Hold,14.60,Low Confidence
7,ABB,Buy,23.76,Low Confidence
8,BAJAJHLDNG,Hold,10.98,Low Confidence
9,TECHM,Hold,12.34,Low Confidence


In [25]:
decision_data["investment_rank"] = (
    decision_data["final_investment_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

decision_data = decision_data.sort_values(
    [
        "investment_rank",
        "normalized_risk_score"
    ]
).reset_index(drop=True)

decision_data[
    [
        "company_id",
        "final_investment_score",
        "recommendation",
        "investment_rank"
    ]
].head(20)

,company_id,final_investment_score,recommendation,investment_rank
0,BEL,95.14,Strong Buy,1
1,HAL,74.30,Buy,2
2,TRENT,65.14,Buy,3
3,INDIGO,64.97,Buy,4
4,ABB,61.88,Buy,5
5,TATAMOTORS,59.82,Hold,6
6,PIDILITIND,59.34,Hold,7
7,NESTLEIND,58.44,Hold,8
8,LT,58.30,Hold,9
9,MARUTI,58.06,Hold,10


In [26]:
final_decision_report = decision_data[
    [
        "company_id",
        score_column,
        "financial_momentum_score",
        "trend_classification",
        "normalized_risk_score",
        "risk_category",
        "primary_risk_driver",
        "quality_score_100",
        "momentum_score_100",
        "safety_score_100",
        "final_investment_score",
        "recommendation",
        "recommendation_confidence",
        "confidence_level",
        "investment_rank"
    ]
].copy()

final_decision_report.head(20)

,company_id,composite_score,financial_momentum_score,trend_classification,normalized_risk_score,risk_category,primary_risk_driver,quality_score_100,momentum_score_100,safety_score_100,final_investment_score,recommendation,recommendation_confidence,confidence_level,investment_rank
0,BEL,73.189325,3,Strong Improvement,6.67,Low Risk,Debt Risk,100.00,85.71,93.33,95.14,Strong Buy,90.28,High Confidence,1
1,HAL,58.206714,0,Declining,13.33,Low Risk,Debt Risk,79.45,42.86,86.67,74.30,Buy,48.60,Moderate Confidence,2
2,TRENT,25.274906,4,Strong Improvement,6.67,Low Risk,Asset Efficiency Risk,34.29,100.00,93.33,65.14,Buy,30.28,Moderate Confidence,3
3,INDIGO,37.526171,1,Improving,6.67,Low Risk,Financial Momentum Risk,51.09,57.14,93.33,64.97,Buy,29.94,Low Confidence,4
4,ABB,31.765038,2,Improving,13.33,Low Risk,Asset Efficiency Risk,43.19,71.43,86.67,61.88,Buy,23.76,Low Confidence,5
5,TATAMOTORS,23.349936,4,Strong Improvement,20.00,Low Risk,Debt Risk,31.65,100.00,80.00,59.82,Hold,19.64,Low Confidence,6
6,PIDILITIND,28.051809,2,Improving,13.33,Low Risk,Asset Efficiency Risk,38.10,71.43,86.67,59.34,Hold,18.68,Low Confidence,7
7,NESTLEIND,28.004046,1,Improving,6.67,Low Risk,Financial Momentum Risk,38.03,57.14,93.33,58.44,Hold,16.88,Low Confidence,8
8,LT,26.538908,2,Improving,13.33,Low Risk,Interest Coverage Risk,36.02,71.43,86.67,58.30,Hold,16.60,Low Confidence,9
9,MARUTI,26.186648,2,Improving,13.33,Low Risk,Asset Efficiency Risk,35.54,71.43,86.67,58.06,Hold,16.12,Low Confidence,10


In [27]:
top_investment_candidates = (
    final_decision_report[
        final_decision_report["recommendation"].isin(
            [
                "Strong Buy",
                "Buy"
            ]
        )
    ]
    .sort_values(
        [
            "final_investment_score",
            "normalized_risk_score"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(20)
    .copy()
)

print(
    "Top Investment Candidates:",
    len(top_investment_candidates)
)

top_investment_candidates

Top Investment Candidates: 5


,company_id,composite_score,financial_momentum_score,trend_classification,normalized_risk_score,risk_category,primary_risk_driver,quality_score_100,momentum_score_100,safety_score_100,final_investment_score,recommendation,recommendation_confidence,confidence_level,investment_rank
0,BEL,73.189325,3,Strong Improvement,6.67,Low Risk,Debt Risk,100.00,85.71,93.33,95.14,Strong Buy,90.28,High Confidence,1
1,HAL,58.206714,0,Declining,13.33,Low Risk,Debt Risk,79.45,42.86,86.67,74.30,Buy,48.60,Moderate Confidence,2
2,TRENT,25.274906,4,Strong Improvement,6.67,Low Risk,Asset Efficiency Risk,34.29,100.00,93.33,65.14,Buy,30.28,Moderate Confidence,3
3,INDIGO,37.526171,1,Improving,6.67,Low Risk,Financial Momentum Risk,51.09,57.14,93.33,64.97,Buy,29.94,Low Confidence,4
4,ABB,31.765038,2,Improving,13.33,Low Risk,Asset Efficiency Risk,43.19,71.43,86.67,61.88,Buy,23.76,Low Confidence,5


In [28]:
investment_warning_list = (
    final_decision_report[
        final_decision_report["recommendation"].isin(
            [
                "Avoid",
                "High Risk"
            ]
        )
    ]
    .sort_values(
        [
            "normalized_risk_score",
            "final_investment_score"
        ],
        ascending=[
            False,
            True
        ]
    )
    .copy()
)

print(
    "Investment Warning Companies:",
    len(investment_warning_list)
)

investment_warning_list.head(20)

Investment Warning Companies: 29


,company_id,composite_score,financial_momentum_score,trend_classification,normalized_risk_score,risk_category,primary_risk_driver,quality_score_100,momentum_score_100,safety_score_100,final_investment_score,recommendation,recommendation_confidence,confidence_level,investment_rank
91,UNIONBANK,3.698431,2,Improving,73.33,High Risk,Debt Risk,4.70,71.43,26.67,24.64,Avoid,50.72,Moderate Confidence,92
90,KOTAKBANK,18.415683,-1,Declining,73.33,High Risk,Debt Risk,24.88,28.57,26.67,26.16,Avoid,47.68,Moderate Confidence,91
88,SHRIRAMFIN,18.716305,-1,Declining,73.33,High Risk,Debt Risk,25.30,28.57,26.67,26.36,Avoid,47.28,Moderate Confidence,89
86,TATASTEEL,25.401622,-3,Strong Decline,66.67,High Risk,Profitability Risk,34.46,0.00,33.33,27.23,Avoid,45.54,Moderate Confidence,87
85,GRASIM,22.670563,-2,Strong Decline,66.67,High Risk,Asset Efficiency Risk,30.72,14.29,33.33,28.22,Avoid,43.56,Moderate Confidence,86
84,PFC,10.961002,1,Improving,66.67,High Risk,Debt Risk,14.66,57.14,33.33,28.76,Avoid,42.48,Moderate Confidence,85
83,JSWENERGY,23.580609,-2,Strong Decline,66.67,High Risk,Asset Efficiency Risk,31.97,14.29,33.33,28.84,Avoid,42.32,Moderate Confidence,84
82,IRFC,11.481723,1,Improving,66.67,High Risk,Debt Risk,15.38,57.14,33.33,29.12,Avoid,41.76,Moderate Confidence,83
79,INDUSINDBK,13.709969,1,Improving,66.67,High Risk,Debt Risk,18.43,57.14,33.33,30.64,Avoid,38.72,Moderate Confidence,80
78,HDFCBANK,13.723837,1,Improving,66.67,High Risk,Debt Risk,18.45,57.14,33.33,30.65,Avoid,38.70,Moderate Confidence,79


In [29]:
recommendation_distribution = (
    final_decision_report["recommendation"]
    .value_counts(dropna=False)
    .rename_axis("recommendation")
    .reset_index(name="company_count")
)

recommendation_distribution

,recommendation,company_count
0,Hold,58
1,Avoid,29
2,Buy,4
3,Strong Buy,1


In [30]:
top_10_investment_ranking = (
    final_decision_report[
        [
            "investment_rank",
            "company_id",
            "final_investment_score",
            "recommendation",
            "normalized_risk_score",
            "trend_classification",
            "confidence_level"
        ]
    ]
    .head(10)
    .copy()
)

top_10_investment_ranking

,investment_rank,company_id,final_investment_score,recommendation,normalized_risk_score,trend_classification,confidence_level
0,1,BEL,95.14,Strong Buy,6.67,Strong Improvement,High Confidence
1,2,HAL,74.30,Buy,13.33,Declining,Moderate Confidence
2,3,TRENT,65.14,Buy,6.67,Strong Improvement,Moderate Confidence
3,4,INDIGO,64.97,Buy,6.67,Improving,Low Confidence
4,5,ABB,61.88,Buy,13.33,Improving,Low Confidence
5,6,TATAMOTORS,59.82,Hold,20.00,Strong Improvement,Low Confidence
6,7,PIDILITIND,59.34,Hold,13.33,Improving,Low Confidence
7,8,NESTLEIND,58.44,Hold,6.67,Improving,Low Confidence
8,9,LT,58.30,Hold,13.33,Improving,Low Confidence
9,10,MARUTI,58.06,Hold,13.33,Improving,Low Confidence


In [31]:
decision_summary = pd.DataFrame(
    {
        "metric": [
            "Companies Analyzed",
            "Strong Buy Companies",
            "Buy Companies",
            "Hold Companies",
            "Avoid Companies",
            "High Risk Companies"
        ],
        "value": [
            final_decision_report["company_id"].nunique(),

            (
                final_decision_report["recommendation"]
                == "Strong Buy"
            ).sum(),

            (
                final_decision_report["recommendation"]
                == "Buy"
            ).sum(),

            (
                final_decision_report["recommendation"]
                == "Hold"
            ).sum(),

            (
                final_decision_report["recommendation"]
                == "Avoid"
            ).sum(),

            (
                final_decision_report["recommendation"]
                == "High Risk"
            ).sum()
        ]
    }
)

decision_summary

,metric,value
0,Companies Analyzed,92
1,Strong Buy Companies,1
2,Buy Companies,4
3,Hold Companies,58
4,Avoid Companies,29
5,High Risk Companies,0


In [32]:
os.makedirs(REPORT_DIR, exist_ok=True)

final_decision_report.to_csv(
    "../reports/day21_investment_decisions.csv",
    index=False
)

top_investment_candidates.to_csv(
    "../reports/day21_top_investment_candidates.csv",
    index=False
)

investment_warning_list.to_csv(
    "../reports/day21_investment_warning_list.csv",
    index=False
)

recommendation_distribution.to_csv(
    "../reports/day21_recommendation_distribution.csv",
    index=False
)

top_10_investment_ranking.to_csv(
    "../reports/day21_top10_investment_ranking.csv",
    index=False
)

decision_summary.to_csv(
    "../reports/day21_decision_summary.csv",
    index=False
)

print("Day 21 investment decision reports saved successfully.")

Day 21 investment decision reports saved successfully.


In [33]:
print("=" * 70)
print("DAY 21 - INVESTMENT DECISION ENGINE")
print("=" * 70)

print("\nCompanies Analyzed:")
print(final_decision_report["company_id"].nunique())

print("\nDECISION MODEL WEIGHTS")
print("Quality Weight  :", QUALITY_WEIGHT)
print("Momentum Weight :", MOMENTUM_WEIGHT)
print("Safety Weight   :", SAFETY_WEIGHT)

print("\nRECOMMENDATION DISTRIBUTION")
print(recommendation_distribution)

print("\nDECISION SUMMARY")
print(decision_summary)

print("\nTOP 10 INVESTMENT RANKING")
print(top_10_investment_ranking)

print("\nTOP INVESTMENT CANDIDATES")
print(
    top_investment_candidates[
        [
            "company_id",
            "final_investment_score",
            "recommendation",
            "normalized_risk_score",
            "investment_rank"
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("DAY 21 COMPLETED SUCCESSFULLY")
print("=" * 70)

DAY 21 - INVESTMENT DECISION ENGINE

Companies Analyzed:
92

DECISION MODEL WEIGHTS
Quality Weight  : 0.5
Momentum Weight : 0.2
Safety Weight   : 0.3

RECOMMENDATION DISTRIBUTION
  recommendation  company_count
0           Hold             58
1          Avoid             29
2            Buy              4
3     Strong Buy              1

DECISION SUMMARY
                 metric  value
0    Companies Analyzed     92
1  Strong Buy Companies      1
2         Buy Companies      4
3        Hold Companies     58
4       Avoid Companies     29
5   High Risk Companies      0

TOP 10 INVESTMENT RANKING
   investment_rank  company_id  final_investment_score recommendation  normalized_risk_score trend_classification     confidence_level
0                1         BEL                   95.14     Strong Buy                   6.67   Strong Improvement      High Confidence
1                2         HAL                   74.30            Buy                  13.33            Declining  Moderate Confi